# Macro Economic Data Viewer

Unified interactive notebook for exploring all economic data hierarchies:
- **CPI** — 322 items from `data/CPI/cpi_hierarchy.csv` (BLS expenditure categories + special aggregates)
- **PCE** — 119 components from `data/PCE/pce_hierarchy.tsv` (BEA NIPA Tables, 6 metric types)
- **CES** — 375 industries from `data/Payrolls/bls_ces_payrolls_hierarchy_comma_safe.csv` (employment, hours, earnings)
- **GDP** — BEA NIPA Table 1.1.x components with optional full PCE detail
- **CPS** — FRED household survey measures (unemployment, LFPR, demographics)

In [ ]:
from datetime import date

import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, clear_output, display

from macro_econ.series.cpi import build_cpi_tree
from macro_econ.series.pce import build_pce_tree
from macro_econ.series.gdp import build_gdp_tree
from macro_econ.series.employment import build_ces_tree, build_cps_tree
from macro_econ.series.loaders import METRIC_OPTIONS
from macro_econ.series.node import SeriesNode
from macro_econ.transforms.changes import (
    level_change, mom_change, mom_annualized, qoq_change,
    qoq_annualized, yoy_change, n_month_annualized,
)
from macro_econ.viz.styles import DEFAULT_LAYOUT

print("Imports ready.")

## 1. Setup: API Clients & Hierarchy Trees

In [ ]:
from macro_econ.clients import FredClient

# Initialize API clients (requires keys in .env)
fred = FredClient()

# Optional: add BLS/BEA clients for broader coverage
# from macro_econ.clients import BlsClient, BeaClient
# bls = BlsClient()
# bea = BeaClient()
bls = None
bea = None

print("API clients ready.")

In [ ]:
# Build all hierarchy trees from data files
trees = {
    "CPI": build_cpi_tree(),
    "PCE": build_pce_tree(),
    "GDP": build_gdp_tree(),
    "CES": build_ces_tree(),
    "CPS": build_cps_tree(),
}

for name, tree in trees.items():
    nodes = list(tree.walk())
    leaves = tree.leaves()
    print(f"{name:4s}: {len(nodes):4d} nodes, {len(leaves):4d} leaves  —  {tree.name}")

## 2. Helper Functions

In [ ]:
# ---------------------------------------------------------------------------
# Transform dispatch
# ---------------------------------------------------------------------------
TRANSFORMS = [
    ("Level", "level"),
    ("MoM %", "mom"),
    ("MoM Annualized %", "mom_ann"),
    ("QoQ %", "qoq"),
    ("QoQ Annualized %", "qoq_ann"),
    ("YoY %", "yoy"),
    ("3m Annualized %", "ann3"),
    ("6m Annualized %", "ann6"),
    ("Level Change", "diff"),
]
_TRANSFORM_LABELS = {v: k for k, v in TRANSFORMS}


def apply_transform(df, transform):
    col = "value"
    dispatch = {
        "level": lambda: df[col],
        "mom": lambda: mom_change(df, col),
        "mom_ann": lambda: mom_annualized(df, col),
        "qoq": lambda: qoq_change(df, col),
        "qoq_ann": lambda: qoq_annualized(df, col),
        "yoy": lambda: yoy_change(df, col, periods=12),
        "ann3": lambda: n_month_annualized(df, n=3, col=col),
        "ann6": lambda: n_month_annualized(df, n=6, col=col),
        "diff": lambda: level_change(df, col=col),
    }
    return dispatch.get(transform, lambda: df[col])()


# ---------------------------------------------------------------------------
# Treemap / Sunburst data builder
# ---------------------------------------------------------------------------
def build_tree_data(root):
    ids, labels, parents, values = [], [], [], []
    def leaf_count(node):
        return 1 if node.is_leaf else sum(leaf_count(c) for c in node.children)
    for node in root.walk():
        path = "/".join(node.path())
        ids.append(path)
        labels.append(node.name)
        parents.append("/".join(node.parent.path()) if node.parent else "")
        values.append(leaf_count(node))
    return ids, labels, parents, values


# ---------------------------------------------------------------------------
# Metric-aware source selection
# ---------------------------------------------------------------------------
def get_metric_source(node, metric):
    if metric:
        for src in node.sources:
            if src.extra.get("metric") == metric:
                return src
    for src in node.sources:
        if src.source == "fred":
            return src
    return node.sources[0] if node.sources else None


# ---------------------------------------------------------------------------
# Data fetching with caching
# ---------------------------------------------------------------------------
data_cache = {}


def fetch_node_data(tree_name, code, start_date=None, end_date=None, metric=""):
    tree = trees[tree_name]
    node = tree.find(code)
    if node is None:
        return None

    cache_key = (tree_name, code, metric)
    if cache_key in data_cache:
        df = data_cache[cache_key].copy()
        if start_date:
            df = df[df.index >= pd.Timestamp(start_date)]
        if end_date:
            df = df[df.index <= pd.Timestamp(end_date)]
        return df

    src = get_metric_source(node, metric)
    if src is None:
        return None

    # Try the matched source's client
    clients = {"fred": fred, "bls": bls, "bea": bea}
    client = clients.get(src.source)
    if client is not None:
        try:
            if src.source == "bea":
                df = client.fetch_series(
                    src.series_id, start_date, end_date,
                    table=src.extra.get("table"),
                    line_number=src.extra.get("line_number"),
                    frequency=src.extra.get("frequency", "M"),
                )
            else:
                df = client.fetch_series(src.series_id, start_date, end_date)
            data_cache[cache_key] = df
            return df
        except Exception:
            pass

    # Fallback: try all sources in priority order
    for source_name in ["fred", "bls", "bea"]:
        c = clients.get(source_name)
        if c is None:
            continue
        for s in node.sources:
            if s.source != source_name:
                continue
            if metric and s.extra.get("metric") and s.extra["metric"] != metric:
                continue
            try:
                if s.source == "bea":
                    df = c.fetch_series(
                        s.series_id, start_date, end_date,
                        table=s.extra.get("table"),
                        line_number=s.extra.get("line_number"),
                        frequency=s.extra.get("frequency", "M"),
                    )
                else:
                    df = c.fetch_series(s.series_id, start_date, end_date)
                data_cache[cache_key] = df
                return df
            except Exception:
                continue
    return None


# ---------------------------------------------------------------------------
# Heatmap cell color
# ---------------------------------------------------------------------------
def color_cell(val, abs_max):
    if pd.isna(val):
        return "background-color: #f5f5f5; color: #aaa"
    if abs_max == 0:
        return "background-color: #ffffff; color: #333"
    ratio = max(-1.0, min(1.0, val / abs_max))
    if ratio > 0:
        r, g, b = 255, int(255 * (1 - ratio * 0.55)), int(255 * (1 - ratio * 0.65))
    elif ratio < 0:
        a = abs(ratio)
        r, g, b = int(255 * (1 - a * 0.65)), int(255 * (1 - a * 0.45)), 255
    else:
        r, g, b = 255, 255, 255
    return f"background-color: rgb({r},{g},{b}); color: #333"


def parse_int_list(text):
    result = []
    for tok in text.split(","):
        tok = tok.strip()
        if tok:
            try:
                result.append(int(tok))
            except ValueError:
                pass
    return result


print("Helpers ready.")

## 3. Pre-built Scenarios

In [ ]:
SCENARIOS = {
    "CPI": {
        "Core vs Headline": ["CPI", "CPI_SA0L1E", "CPI_SASLE"],
        "Shelter Components": ["CPI_SAH1", "CPI_SEHC", "CPI_SEHA", "CPI_SEHB"],
        "Food": ["CPI_SAF1", "CPI_SAF11", "CPI_SEFV", "CPI_SAF2"],
        "Energy": ["CPI_SA0E", "CPI_SETB", "CPI_SAH21", "CPI_SEHF01", "CPI_SEHF02"],
        "Transportation": ["CPI_SAT", "CPI_SETA01", "CPI_SETA02", "CPI_SETB", "CPI_SETE", "CPI_SETG01"],
        "Medical": ["CPI_SAM", "CPI_SAM1", "CPI_SAM2", "CPI_SEME"],
        "Major Groups": ["CPI_SAF", "CPI_SAH", "CPI_SAA", "CPI_SAT", "CPI_SAM", "CPI_SAR", "CPI_SAE", "CPI_SAG"],
    },
    "PCE": {
        "Goods vs Services": ["PCE", "PCE_GOODS", "PCE_SERVICES"],
        "Durable vs Nondurable": ["PCE_DURABLE_GOODS", "PCE_NONDURABLE_GOODS"],
        "Services Detail": [
            "PCE_HOUSING_AND_UTILITIES", "PCE_HEALTH_CARE",
            "PCE_TRANSPORTATION_SERVICES", "PCE_RECREATION_SERVICES",
            "PCE_FOOD_SERVICES_AND_ACCOMMODATIONS", "PCE_FINANCIAL_SERVICES_AND_INSURANCE",
        ],
        "Motor Vehicles": [
            "PCE_MOTOR_VEHICLES_AND_PARTS", "PCE_NEW_MOTOR_VEHICLES",
            "PCE_NET_USED_MOTOR_VEHICLES", "PCE_MOTOR_VEHICLE_PARTS_AND_ACCESSORIES",
        ],
    },
    "GDP": {
        "Main Components (C+I+G+NX)": ["GDP_C", "GDP_I", "GDP_NX", "GDP_G"],
        "Investment Detail": ["GDP_I_FIXED", "GDP_I_NONRES", "GDP_I_RES", "GDP_I_INV"],
        "Nonresidential": ["GDP_I_STRUCT", "GDP_I_EQUIP", "GDP_I_IP"],
        "Government": ["GDP_G_FED", "GDP_G_DEF", "GDP_G_NONDEF", "GDP_G_SL"],
        "Trade": ["GDP_X", "GDP_M", "GDP_X_GOODS", "GDP_X_SVC", "GDP_M_GOODS", "GDP_M_SVC"],
        "Price Measures": ["GDP_DEFL", "GDP_CHAIN_PI"],
    },
    "CES": {
        "Major Sectors": ["CES_05_000000", "CES_06_000000", "CES_08_000000", "CES_90_000000"],
        "Goods Producing": ["CES_10_000000", "CES_20_000000", "CES_30_000000"],
        "Service Providing": [
            "CES_40_000000", "CES_50_000000", "CES_55_000000", "CES_60_000000",
            "CES_65_000000", "CES_70_000000", "CES_80_000000",
        ],
        "Manufacturing": ["CES_31_000000", "CES_32_000000"],
    },
    "CPS": {
        "Headline": ["UR", "LFPR", "EPOP"],
        "Alternative Unemployment (U1-U6)": ["U1", "U2", "U3", "U4", "U5", "U6"],
        "By Age": ["UR_16_19", "UR_20_24", "UR_25_54", "UR_55_PLUS"],
        "By Gender": ["UR_MEN", "UR_WOMEN"],
        "By Race": ["UR_WHITE", "UR_BLACK", "UR_HISPANIC", "UR_ASIAN"],
        "Prime Age": ["UR_25_54", "LFPR_25_54", "EPOP_25_54"],
    },
}

TABLE_CSS = """
<style>
.macro-viewer-table th {
    background-color: #2c3e50 !important; color: white !important;
    text-align: center !important; padding: 6px 8px !important;
    font-size: 11px !important; border: 1px solid #34495e !important; white-space: nowrap;
}
.macro-viewer-table th.row_heading {
    background-color: #34495e !important; color: white !important;
    text-align: left !important; font-weight: bold !important; min-width: 180px !important;
}
.macro-viewer-table td {
    text-align: center !important; font-size: 11px !important;
    padding: 4px 6px !important; border: 1px solid #eee !important; white-space: nowrap;
}
.macro-viewer-table { border-collapse: collapse !important; width: 100% !important; }
</style>
"""

print(f"Scenarios defined for: {', '.join(SCENARIOS.keys())}")

---
## 4. Interactive Viewer

**Run the cell below** to launch the unified viewer. Use the controls to:
1. **Select an index** (CPI, PCE, GDP, CES, CPS)
2. **Choose a metric** (e.g., CES: employment vs. earnings; PCE: price index vs. nominal)
3. **Explore the hierarchy** via Treemap or Sunburst
4. **Drilldown** into a single series with transforms and moving averages
5. **Compare** multiple series in a color-coded heatmap table

In [ ]:
# ====================================================================
# Top bar: Index, Metric, Hierarchy mode
# ====================================================================
w_index = widgets.Dropdown(
    options=list(trees.keys()), value="CPI",
    description="Index:", style={"description_width": "auto"},
    layout=widgets.Layout(width="140px"),
)
w_metric = widgets.Dropdown(
    options=[("Default", "")], value="",
    description="Metric:", style={"description_width": "auto"},
    layout=widgets.Layout(width="260px"),
)
w_hier_mode = widgets.Dropdown(
    options=["Treemap", "Sunburst"], value="Treemap",
    description="Hierarchy:", style={"description_width": "auto"},
    layout=widgets.Layout(width="190px"),
)
btn_hierarchy = widgets.Button(
    description="Show Hierarchy", button_style="info", icon="sitemap",
    layout=widgets.Layout(width="160px"),
)
top_bar = widgets.HBox(
    [w_index, w_metric, w_hier_mode, btn_hierarchy],
    layout=widgets.Layout(
        justify_content="flex-start", padding="8px 10px",
        gap="10px", border="1px solid #ccc", margin="0 0 6px 0",
    ),
)

# ====================================================================
# Left panel: Single Drilldown
# ====================================================================
w_drill_series = widgets.Dropdown(
    options=[], description="Series:",
    style={"description_width": "50px"}, layout=widgets.Layout(width="100%"),
)
w_drill_transform = widgets.Dropdown(
    options=TRANSFORMS, value="level", description="Type:",
    style={"description_width": "50px"}, layout=widgets.Layout(width="100%"),
)
w_drill_start = widgets.IntText(
    value=2010, description="Start Yr:",
    style={"description_width": "60px"}, layout=widgets.Layout(width="100%"),
)
w_drill_end = widgets.IntText(
    value=2026, description="End Yr:",
    style={"description_width": "60px"}, layout=widgets.Layout(width="100%"),
)
w_drill_avg = widgets.Text(
    value="3, 6, 12", description="Avg Periods:",
    style={"description_width": "80px"}, layout=widgets.Layout(width="100%"),
    placeholder="e.g. 3, 6, 12",
)
btn_drilldown = widgets.Button(
    description="Run Drilldown", button_style="primary", icon="line-chart",
    layout=widgets.Layout(width="100%"),
)
left_panel = widgets.VBox(
    [
        widgets.HTML("<div style='font-weight:bold; padding:2px 0 6px 0; border-bottom:1px solid #ccc; margin-bottom:6px'>Single Series Drilldown</div>"),
        w_drill_series, w_drill_transform,
        widgets.HBox([w_drill_start, w_drill_end]),
        w_drill_avg, btn_drilldown,
    ],
    layout=widgets.Layout(width="28%", padding="8px", border="1px solid #ccc", margin="0 4px 0 0"),
)

# ====================================================================
# Middle panel: Multi-Series Select
# ====================================================================
w_scenario = widgets.Dropdown(
    options=[], description="Scenario:",
    style={"description_width": "65px"}, layout=widgets.Layout(width="100%"),
)
btn_scenario = widgets.Button(
    description="Load Scenario", button_style="", icon="download",
    layout=widgets.Layout(width="100%"),
)
w_multi = widgets.SelectMultiple(
    options=[], description="",
    layout=widgets.Layout(width="100%", height="220px"),
)
mid_panel = widgets.VBox(
    [
        widgets.HTML("<div style='font-weight:bold; padding:2px 0 6px 0; border-bottom:1px solid #ccc; margin-bottom:6px'>Multi-Series Select</div>"),
        w_scenario, btn_scenario,
        widgets.HTML("<small style='color:#666'>Ctrl+Click for multiple</small>"),
        w_multi,
    ],
    layout=widgets.Layout(width="36%", padding="8px", border="1px solid #ccc", margin="0 4px"),
)

# ====================================================================
# Right panel: Table Controls
# ====================================================================
w_tbl_transform = widgets.Dropdown(
    options=TRANSFORMS, value="mom", description="Type:",
    style={"description_width": "50px"}, layout=widgets.Layout(width="100%"),
)
w_tbl_start = widgets.DatePicker(
    description="Start:", value=date(2022, 1, 1),
    style={"description_width": "50px"}, layout=widgets.Layout(width="100%"),
)
w_tbl_end = widgets.DatePicker(
    description="End:", value=date.today(),
    style={"description_width": "50px"}, layout=widgets.Layout(width="100%"),
)
w_tbl_avg = widgets.Text(
    value="3, 6, 12", description="Averages:",
    style={"description_width": "65px"}, layout=widgets.Layout(width="100%"),
    placeholder="e.g. 3, 6, 12",
)
w_tbl_cols = widgets.IntSlider(
    value=18, min=6, max=36, step=1, description="Columns:",
    style={"description_width": "65px"}, layout=widgets.Layout(width="100%"),
)
btn_table = widgets.Button(
    description="Run Table", button_style="primary", icon="table",
    layout=widgets.Layout(width="100%"),
)
right_panel = widgets.VBox(
    [
        widgets.HTML("<div style='font-weight:bold; padding:2px 0 6px 0; border-bottom:1px solid #ccc; margin-bottom:6px'>Table Controls</div>"),
        w_tbl_transform, w_tbl_start, w_tbl_end,
        w_tbl_avg, w_tbl_cols, btn_table,
    ],
    layout=widgets.Layout(width="36%", padding="8px", border="1px solid #ccc", margin="0 0 0 4px"),
)

# ====================================================================
# Output tabs
# ====================================================================
out_drill = widgets.Output()
out_table = widgets.Output()
out_hier = widgets.Output()
tabs = widgets.Tab(children=[out_drill, out_table, out_hier])
tabs.set_title(0, "Single Drilldown")
tabs.set_title(1, "Multi-Series Table")
tabs.set_title(2, "Index Hierarchy")

controls_row = widgets.HBox(
    [left_panel, mid_panel, right_panel],
    layout=widgets.Layout(margin="0 0 6px 0"),
)


# ====================================================================
# Callbacks
# ====================================================================
def on_index_change(_change):
    idx = w_index.value
    tree = trees[idx]

    # Metric dropdown
    metric_opts = METRIC_OPTIONS.get(idx, {})
    if metric_opts:
        w_metric.options = [("Default", "")] + [(label, code) for code, label in metric_opts.items()]
        w_metric.value = ""
        w_metric.layout.display = ""
    else:
        w_metric.options = [("Default", "")]
        w_metric.value = ""
        w_metric.layout.display = "none"

    # Drilldown series list
    opts = [(f"{n.name} [{n.code}]", n.code) for n in tree.walk() if n.sources]
    w_drill_series.options = opts
    if opts:
        w_drill_series.value = opts[0][1]

    # Multi-select list
    branch = [(f"\u25b6 {n.name} [{n.code}]", n.code) for n in tree.walk() if not n.is_leaf and n.sources]
    leaf = [(f"  {n.name} [{n.code}]", n.code) for n in tree.leaves()]
    w_multi.options = branch + leaf

    # Scenarios
    scenarios = SCENARIOS.get(idx, {})
    w_scenario.options = list(scenarios.keys()) or ["(none)"]


def on_load_scenario(_btn):
    codes = SCENARIOS.get(w_index.value, {}).get(w_scenario.value, [])
    if not codes:
        return
    available = {v for _, v in w_multi.options}
    w_multi.value = tuple(c for c in codes if c in available)


def on_show_hierarchy(_btn):
    idx = w_index.value
    tree = trees[idx]
    mode = w_hier_mode.value
    ids, labels, parents, values = build_tree_data(tree)

    with out_hier:
        clear_output(wait=True)
        if mode == "Treemap":
            trace = go.Treemap(
                ids=ids, labels=labels, parents=parents, values=values,
                branchvalues="total", textinfo="label",
                hovertemplate="<b>%{label}</b><extra></extra>",
                marker=dict(colorscale="Blues", line=dict(width=1, color="white")),
            )
        else:
            trace = go.Sunburst(
                ids=ids, labels=labels, parents=parents, values=values,
                branchvalues="total",
                hovertemplate="<b>%{label}</b><extra></extra>",
            )
        fig = go.Figure(trace)
        fig.update_layout(title=f"{tree.name} — {mode} Hierarchy", height=650, margin=dict(l=10, r=10, t=50, b=10))
        fig.show()
    tabs.selected_index = 2


def on_run_drilldown(_btn):
    code = w_drill_series.value
    transform = w_drill_transform.value
    start_yr, end_yr = w_drill_start.value, w_drill_end.value
    avg_periods = parse_int_list(w_drill_avg.value)
    metric = w_metric.value
    idx = w_index.value

    start_date = f"{start_yr}-01-01"
    end_date = f"{end_yr}-12-31"
    fetch_start = f"{start_yr - 2}-01-01"

    with out_drill:
        clear_output(wait=True)
        df = fetch_node_data(idx, code, fetch_start, end_date, metric=metric)
        if df is None or df.empty:
            print(f"No data for {code}. Check API client configuration.")
            return

        node = trees[idx].find(code)
        name = node.name if node else code
        transform_label = _TRANSFORM_LABELS.get(transform, transform)
        metric_labels = METRIC_OPTIONS.get(idx, {})
        metric_suffix = f" ({metric_labels[metric]})" if metric and metric in metric_labels else ""

        series = apply_transform(df, transform)
        series = series[series.index >= pd.Timestamp(start_date)]

        fig = go.Figure()
        fig.add_trace(go.Scatter(x=series.index, y=series.values, name=name, mode="lines", line=dict(color="#1f77b4", width=2)))
        ma_colors = ["#ff7f0e", "#2ca02c", "#d62728"]
        for i, n in enumerate(avg_periods[:3]):
            ma = series.rolling(window=n, min_periods=1).mean()
            fig.add_trace(go.Scatter(x=ma.index, y=ma.values, name=f"{n}-period MA", mode="lines", line=dict(color=ma_colors[i % 3], dash="dash")))
        fig.update_layout(title=f"{name}{metric_suffix} — {transform_label}", yaxis_title=transform_label, height=450, **DEFAULT_LAYOUT)
        fig.show()

        recent = series.dropna().tail(24).to_frame(name=name).round(2)
        display(HTML(f"<h4>Last {len(recent)} Observations ({transform_label})</h4>"))
        styled = recent.style.format("{:.2f}").set_properties(**{"text-align": "center", "font-size": "12px", "padding": "3px 8px"})
        display(styled)
    tabs.selected_index = 0


def on_run_table(_btn):
    selected_codes = list(w_multi.value)
    transform = w_tbl_transform.value
    start_dt, end_dt = w_tbl_start.value, w_tbl_end.value
    avg_periods = parse_int_list(w_tbl_avg.value)
    max_cols = w_tbl_cols.value
    metric = w_metric.value
    idx = w_index.value

    if not selected_codes:
        with out_table:
            clear_output(wait=True)
            print("Select at least one series in the multi-select list.")
        return

    start_date = start_dt.isoformat() if start_dt else "2018-01-01"
    end_date = end_dt.isoformat() if end_dt else None
    transform_label = _TRANSFORM_LABELS.get(transform, transform)

    with out_table:
        clear_output(wait=True)
        print(f"Fetching {len(selected_codes)} series ...")

        tree = trees[idx]
        metric_labels = METRIC_OPTIONS.get(idx, {})
        metric_suffix = f" ({metric_labels[metric]})" if metric and metric in metric_labels else ""

        all_series = {}
        for code in selected_codes:
            node = tree.find(code)
            label = node.name if node else code
            fetch_start = f"{int(start_date[:4]) - 2}-01-01" if start_date else None
            df = fetch_node_data(idx, code, fetch_start, end_date, metric=metric)
            if df is not None and not df.empty:
                s = apply_transform(df, transform)
                if start_date:
                    s = s[s.index >= pd.Timestamp(start_date)]
                if end_date:
                    s = s[s.index <= pd.Timestamp(end_date)]
                all_series[label] = s

        if not all_series:
            clear_output(wait=True)
            print("No data fetched. Ensure at least one API client is configured (fred=, bls=, bea=).")
            return

        combined = pd.DataFrame(all_series).T
        combined.columns = pd.to_datetime(combined.columns)
        combined = combined.sort_index(axis=1)
        if combined.shape[1] > max_cols:
            combined = combined.iloc[:, -max_cols:]
        combined.columns = [c.strftime("%b %Y") for c in combined.columns]

        for n in sorted(avg_periods, reverse=True):
            if combined.shape[1] >= n:
                combined.insert(0, f"{n}mo\nAvg", combined.iloc[:, -min(n, combined.shape[1]):].mean(axis=1))

        clear_output(wait=True)
        display(HTML(TABLE_CSS))
        display(HTML(f"<h3 style='margin:0 0 8px 0'>{idx}{metric_suffix} — Multi-Series Comparison ({transform_label})</h3>"))

        numeric_vals = combined.select_dtypes(include=[np.number])
        if numeric_vals.empty:
            display(combined.round(2))
        else:
            vmin, vmax = numeric_vals.min().min(), numeric_vals.max().max()
            abs_max = max(abs(vmin), abs(vmax)) if pd.notna(vmin) and pd.notna(vmax) else 1.0
            styled = (
                combined.round(2).style
                .map(lambda v: color_cell(v, abs_max))
                .format("{:.2f}", na_rep="\u2014")
                .set_table_attributes('class="macro-viewer-table"')
            )
            display(styled)

        display(HTML("<h4 style='margin:12px 0 4px 0'>Heatmap View</h4>"))
        fig = go.Figure(data=go.Heatmap(
            z=combined.values, x=list(combined.columns), y=list(combined.index),
            colorscale="RdBu_r", zmid=0,
            text=combined.round(1).values, texttemplate="%{text}", hoverongaps=False,
        ))
        fig.update_layout(
            title=f"{idx}{metric_suffix} — {transform_label}",
            height=max(300, len(combined) * 40),
            margin=dict(l=220, r=30, t=60, b=40),
            yaxis=dict(autorange="reversed"),
        )
        fig.show()
    tabs.selected_index = 1


# Wire callbacks
w_index.observe(on_index_change, names="value")
btn_hierarchy.on_click(on_show_hierarchy)
btn_scenario.on_click(on_load_scenario)
btn_drilldown.on_click(on_run_drilldown)
btn_table.on_click(on_run_table)

# Initialize
on_index_change(None)

# Display
display(widgets.VBox([top_bar, controls_row, tabs], layout=widgets.Layout(width="100%")))

---
## 5. Hierarchy Explorer

Print the full tree for any index to see all available nodes and their codes.

In [ ]:
# Change the index name to explore different hierarchies
index_to_explore = "CPI"  # Options: CPI, PCE, GDP, CES, CPS
max_lines = 80  # Set to None to print all

tree_str = trees[index_to_explore].print_tree()
lines = tree_str.split("\n")
print(f"=== {index_to_explore} Hierarchy ({len(lines)} total nodes) ===")
print("\n".join(lines[:max_lines]))
if max_lines and len(lines) > max_lines:
    print(f"\n... ({len(lines) - max_lines} more nodes)")

---
## 6. Direct Programmatic Access

Use the trees and helpers directly for custom analysis outside the widget UI.

In [ ]:
# Example: find a node and inspect its sources
node = trees["CPI"].find("CPI_SAH1")  # Shelter
if node:
    print(f"Node: {node.name} [{node.code}]")
    print(f"Level: {node.level}")
    print(f"Children: {len(node.children)}")
    print(f"Sources:")
    for src in node.sources:
        print(f"  {src.source}: {src.series_id}  metric={src.extra.get('metric', '-')}")

In [ ]:
# Example: fetch and plot a series directly
from macro_econ.viz.charts import line_chart, recession_shading

cpi_all = fred.fetch_series("CPIAUCSL", start_date="2000-01-01")
cpi_core = fred.fetch_series("CPILFESL", start_date="2000-01-01")

fig = line_chart(
    {"CPI-U YoY": yoy_change(cpi_all, 12), "Core CPI YoY": yoy_change(cpi_core, 12)},
    "CPI: Headline vs Core (YoY %)", yaxis_title="Percent",
)
fig = recession_shading(fig)
fig.show()

---
## Tips

| Feature | How to Use |
|---|---|
| **Metric switching** | Select CES then pick "Avg Hourly Earnings ($)" to see wage data instead of employment |
| **PCE price vs level** | Select PCE then pick "Price Index" to compare PCE deflator components |
| **CPI SA vs NSA** | Select CPI then toggle between Seasonally Adjusted and Not Seasonally Adjusted |
| **Average columns** | 3mo, 6mo, 12mo averages are prepended to the table for trend analysis |
| **Column slider** | Adjust the Columns slider to show more or fewer time periods |
| **Plotly interactivity** | Hover for values, zoom, pan, and export in all charts |
| **Metric dropdown hidden** | For GDP and CPS, the Metric dropdown is hidden (no multi-metric dimension) |